# DINOv2-Base Diverse-Frame Pruning on MELD

This notebook retains 20% of the frames selected by centroid-seeded max-min cosine diversity in frozen DINOv2-base embeddings. The selected original frames are passed through Qwen2.5-VL-3B, followed by evaluation with the previous MLP and retraining a matched MLP.

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
INDEX_DIR = CODE / "outputs/indexes"

FULL_ROOT = CODE / "embeddings/qwen2_5_vl_3b_shared"
CHECKPOINT = CODE / "outputs/qwen2_5_vl_3b_shared/best_mlp.pt"
BASELINE_METRICS = CODE / "outputs/qwen2_5_vl_3b_shared/metrics.json"

FRAME_ROOT = Path("/kaggle/working/embeddings/qwen2_5_vl_3b_dinov2_base_frames_020")
FRAME_CHUNKS = Path("/kaggle/working/embeddings/qwen2_5_vl_3b_dinov2_base_frames_020_chunks")
FRAME_RESULTS = Path("/kaggle/working/outputs/qwen2_5_vl_3b_dinov2_base_frames_020")
SMOKE_OUTPUT = Path("/kaggle/working/smoke_dinov2_base_frames_020.pt")

RETAIN_RATIO = 0.20
FRAME_ENCODER = "facebook/dinov2-base"
FRAME_ENCODER_DEVICE = "cuda"
FRAME_ENCODER_BATCH_SIZE = 16

for path in [FRAME_ROOT, FRAME_CHUNKS, FRAME_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

extractor = CODE / "scripts/extract_qwen_shared_embeddings.py"
runner = CODE / "scripts/run_qwen_shared_chunks.py"
selector = CODE / "scripts/frame_diversity.py"

assert CODE.exists(), f"Missing Kaggle code dataset: {CODE}"
assert extractor.exists() and runner.exists() and selector.exists()
assert "--frame-selection" in extractor.read_text()
assert 'DEFAULT_ENCODER_MODEL = "facebook/dinov2-base"' in selector.read_text()
assert CHECKPOINT.exists(), f"Missing baseline checkpoint: {CHECKPOINT}"
assert BASELINE_METRICS.exists(), f"Missing baseline metrics: {BASELINE_METRICS}"

print("Code:", CODE)
print("Frame encoder:", FRAME_ENCODER)
print("Frame retain ratio:", RETAIN_RATIO)

In [ ]:
# Install the repository dependencies, then pin the versions used by the earlier Qwen experiment.
!pip install -q -r {CODE}/requirements.txt
!pip install -q "transformers==5.0.0" "qwen-vl-utils[decord]==0.0.14"

In [ ]:
import json
import os

import pandas as pd
import torch
import transformers

print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running extraction.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

for split in ["train", "dev", "test"]:
    index_path = INDEX_DIR / f"meld_{split}_index.csv"
    assert index_path.exists(), index_path
    frame = pd.read_csv(index_path)
    available = int(frame["video_exists"].astype(bool).sum()) if "video_exists" in frame else len(frame)
    print(f"{split}: rows={len(frame)}, available_videos={available}")

## Smoke test

This two-sample run downloads and loads both models, exercises DINOv2 selection on CUDA, and verifies the saved frame metadata before the full extraction begins. After one successful execution, this cell may be skipped when resuming the notebook.

In [ ]:
!python {extractor} \
  --index-csv {INDEX_DIR}/meld_dev_index.csv \
  --output-pt {SMOKE_OUTPUT} \
  --limit 2 \
  --fps 6 \
  --min-frames 4 \
  --max-frames 64 \
  --frame-size 224 \
  --pooling last \
  --prompt-style emotion_task \
  --modality-mode video_text \
  --frame-selection diverse \
  --frame-retain-ratio {RETAIN_RATIO} \
  --frame-encoder-model {FRAME_ENCODER} \
  --frame-encoder-device {FRAME_ENCODER_DEVICE} \
  --frame-encoder-batch-size {FRAME_ENCODER_BATCH_SIZE} \
  --save-dtype float32 \
  --gc-every 1 \
  --batch-save-every 1

smoke = torch.load(SMOKE_OUTPUT, map_location="cpu", weights_only=False)
assert smoke["config"]["frame_selection"] == "diverse"
assert abs(float(smoke["config"]["frame_retain_ratio"]) - RETAIN_RATIO) < 1e-12
assert smoke["config"]["frame_encoder_model"] == FRAME_ENCODER
assert len(smoke["metadata"]) > 0
for row in smoke["metadata"]:
    indices = row["selected_frame_indices"]
    assert indices == sorted(indices)
    assert len(indices) == len(set(indices)) == row["retained_frame_count"]
display(pd.DataFrame(smoke["metadata"])[[
    "sample_id", "candidate_frame_count", "retained_frame_count",
    "actual_frame_retain_ratio", "frame_selection_seconds",
    "video_token_count", "selected_frame_indices",
]])

## Extract train, development, and test embeddings

Each chunk runs in a fresh process so GPU and decoder memory are released between chunks. Existing complete chunks with the same selector configuration are skipped.

In [ ]:
for split in ["dev", "test", "train"]:
    !python {runner} \
      --index-csv {INDEX_DIR}/meld_{split}_index.csv \
      --chunks-dir {FRAME_CHUNKS}/{split} \
      --chunk-prefix {split} \
      --chunk-size 500 \
      --fps 6 \
      --min-frames 4 \
      --max-frames 64 \
      --frame-size 224 \
      --pooling last \
      --prompt-style emotion_task \
      --modality-mode video_text \
      --frame-selection diverse \
      --frame-retain-ratio {RETAIN_RATIO} \
      --frame-encoder-model {FRAME_ENCODER} \
      --frame-encoder-device {FRAME_ENCODER_DEVICE} \
      --frame-encoder-batch-size {FRAME_ENCODER_BATCH_SIZE} \
      --save-dtype float32 \
      --gc-every 5 \
      --batch-save-every 25 \
      --skip-existing-complete

In [ ]:
# Merge each split after all of its chunks complete.
for split in ["train", "dev", "test"]:
    !python {CODE}/scripts/merge_embedding_chunks.py \
      --chunks-dir {FRAME_CHUNKS}/{split} \
      --output-pt {FRAME_ROOT}/meld_{split}.pt \
      --pattern "{split}_*.pt"

## Validate extraction and summarize efficiency

The Qwen visual-token ratio is computed on sample IDs available in both the full baseline and frame-pruned extraction. Selector time includes only DINOv2 preprocessing, encoding, and diverse selection for each sample.

In [ ]:
import numpy as np

efficiency_rows = []

for split in ["train", "dev", "test"]:
    pruned = torch.load(FRAME_ROOT / f"meld_{split}.pt", map_location="cpu", weights_only=False)
    full = torch.load(FULL_ROOT / f"meld_{split}.pt", map_location="cpu", weights_only=False)

    x = pruned["embeddings"].float()
    assert torch.isfinite(x).all(), f"Non-finite embeddings in {split}"
    assert len(pruned["sample_ids"]) == x.shape[0] == pruned["labels"].shape[0]

    metadata = pruned["metadata"]
    for row in metadata:
        indices = row["selected_frame_indices"]
        assert indices == sorted(indices)
        assert len(indices) == len(set(indices)) == int(row["retained_frame_count"])
        expected = max(1, round(int(row["candidate_frame_count"]) * RETAIN_RATIO))
        assert int(row["retained_frame_count"]) == expected

    full_meta = {str(sid): meta for sid, meta in zip(full["sample_ids"], full["metadata"])}
    pruned_meta = {str(sid): meta for sid, meta in zip(pruned["sample_ids"], metadata)}
    aligned_ids = sorted(set(full_meta) & set(pruned_meta))
    full_tokens = [int(full_meta[sid]["video_token_count"]) for sid in aligned_ids]
    pruned_tokens = [int(pruned_meta[sid]["video_token_count"]) for sid in aligned_ids]
    selection_seconds = np.array([float(row["frame_selection_seconds"]) for row in metadata])
    candidate_frames = np.array([int(row["candidate_frame_count"]) for row in metadata])
    retained_frames = np.array([int(row["retained_frame_count"]) for row in metadata])

    efficiency_rows.append({
        "split": split,
        "samples": len(pruned["sample_ids"]),
        "errors": len(pruned.get("errors", [])),
        "embedding_shape": str(tuple(x.shape)),
        "mean_candidate_frames": candidate_frames.mean(),
        "mean_retained_frames": retained_frames.mean(),
        "actual_frame_ratio": retained_frames.sum() / candidate_frames.sum(),
        "aligned_samples": len(aligned_ids),
        "mean_full_qwen_tokens": np.mean(full_tokens),
        "mean_pruned_qwen_tokens": np.mean(pruned_tokens),
        "qwen_token_ratio": np.sum(pruned_tokens) / np.sum(full_tokens),
        "mean_selector_seconds": selection_seconds.mean(),
        "p95_selector_seconds": np.percentile(selection_seconds, 95),
    })

efficiency = pd.DataFrame(efficiency_rows)
display(efficiency.round(4))
FRAME_RESULTS.mkdir(parents=True, exist_ok=True)
efficiency.to_csv(FRAME_RESULTS / "efficiency_summary.csv", index=False)

## Evaluate the previous MLP

This measures representation shift using the classifier and standardization statistics trained on unpruned Qwen embeddings.

In [ ]:
PREVIOUS_MLP_RESULTS = FRAME_RESULTS / "previous_mlp"

for split in ["dev", "test"]:
    !python {CODE}/scripts/evaluate_mlp.py \
      --checkpoint {CHECKPOINT} \
      --embeddings-pt {FRAME_ROOT}/meld_{split}.pt \
      --output-dir {PREVIOUS_MLP_RESULTS} \
      --split-name {split} \
      --device cuda

## Retrain the MLP on frame-pruned embeddings

The architecture and hyperparameters match the full baseline. Checkpoint selection uses development macro F1.

In [ ]:
NEW_MLP_RESULTS = FRAME_RESULTS / "new_mlp"

!python {CODE}/scripts/train_mlp.py \
  --train-pt {FRAME_ROOT}/meld_train.pt \
  --dev-pt {FRAME_ROOT}/meld_dev.pt \
  --test-pt {FRAME_ROOT}/meld_test.pt \
  --output-dir {NEW_MLP_RESULTS} \
  --hidden-dim 512 \
  --dropout 0.3 \
  --epochs 50 \
  --batch-size 64 \
  --learning-rate 3e-4 \
  --weight-decay 1e-4 \
  --seed 42 \
  --device cuda

## Final performance comparison

In [ ]:
baseline = json.loads(BASELINE_METRICS.read_text())
old_dev = json.loads((PREVIOUS_MLP_RESULTS / "dev_metrics.json").read_text())
old_test = json.loads((PREVIOUS_MLP_RESULTS / "test_metrics.json").read_text())
new = json.loads((NEW_MLP_RESULTS / "metrics.json").read_text())

def metric_row(representation, split, values):
    return {
        "Representation": representation,
        "Split": split,
        "Accuracy": values["accuracy"],
        "Macro F1": values["macro_f1"],
        "Weighted F1": values["weighted_f1"],
    }

rows = [
    metric_row("Full frames + baseline MLP", "Dev", baseline["best_dev"]),
    metric_row("Full frames + baseline MLP", "Test", baseline["test"]),
    metric_row("DINOv2-base diverse 0.20 + previous MLP", "Dev", old_dev),
    metric_row("DINOv2-base diverse 0.20 + previous MLP", "Test", old_test),
    metric_row("DINOv2-base diverse 0.20 + new MLP", "Dev", new["best_dev"]),
    metric_row("DINOv2-base diverse 0.20 + new MLP", "Test", new["test"]),
]
results = pd.DataFrame(rows)
display(results.round(4))

baseline_test = results[(results["Representation"] == "Full frames + baseline MLP") & (results["Split"] == "Test")].iloc[0]
new_test = results[(results["Representation"] == "DINOv2-base diverse 0.20 + new MLP") & (results["Split"] == "Test")].iloc[0]
deltas = pd.DataFrame([{
    "Comparison": "DINOv2-base diverse 0.20 new MLP minus full baseline",
    "Accuracy delta": new_test["Accuracy"] - baseline_test["Accuracy"],
    "Macro F1 delta": new_test["Macro F1"] - baseline_test["Macro F1"],
    "Weighted F1 delta": new_test["Weighted F1"] - baseline_test["Weighted F1"],
}])
display(deltas.round(4))

results.to_csv(FRAME_RESULTS / "metrics_summary.csv", index=False)
deltas.to_csv(FRAME_RESULTS / "baseline_deltas.csv", index=False)

In [ ]:
# Inspect overfitting and identify the selected epoch.
import matplotlib.pyplot as plt

history = pd.read_csv(NEW_MLP_RESULTS / "training_history.csv")
best_epoch = int(history.loc[history["dev_macro_f1"].idxmax(), "epoch"])
print("Best epoch by dev macro F1:", best_epoch)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="train loss")
axes[0].plot(history["epoch"], history["dev_loss"], label="dev loss")
axes[0].axvline(best_epoch, color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(history["epoch"], history["dev_macro_f1"], label="dev macro F1")
axes[1].axvline(best_epoch, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Labeled confusion matrices and per-class reports for the retrained probe.
labels = new.get("label_names", ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"])

def labeled_confusion_matrix(split_metrics):
    return pd.DataFrame(
        split_metrics["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

print("Development confusion matrix")
display(labeled_confusion_matrix(new["best_dev"]))
print("Test confusion matrix")
display(labeled_confusion_matrix(new["test"]))
print("Development classification report")
display(pd.DataFrame(new["best_dev"]["classification_report"]).T.round(4))
print("Test classification report")
display(pd.DataFrame(new["test"]["classification_report"]).T.round(4))

In [ ]:
# Bundle compact result artifacts for download. Embedding tensors remain in FRAME_ROOT.
import shutil

archive_base = Path("/kaggle/working/qwen2_5_vl_3b_dinov2_base_frames_020_results")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=FRAME_RESULTS)
print("Results directory:", FRAME_RESULTS)
print("Embeddings directory:", FRAME_ROOT)
print("Downloadable results archive:", archive_path)
print("Result files:")
for path in sorted(FRAME_RESULTS.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(FRAME_RESULTS))